In [1]:
pip install pandas numpy jupyterlab scikit-learn torch

Note: you may need to restart the kernel to use updated packages.


In [23]:
import pandas as pd
train=pd.read_csv('./dataset/train.csv')
test=pd.read_csv('./dataset/test.csv')

# Dropping id from test dataset
id_test = test['id']
x_test = test.drop(['id'], axis=1, inplace=True)

train.head()

,id,RhythmScore,AudioLoudness,VocalContent,AcousticQuality,InstrumentalScore,LivePerformanceLikelihood,MoodScore,TrackDurationMs,Energy,BeatsPerMinute
0,0,0.603610,-7.636942,0.023500,0.000005,0.000001,0.051385,0.409866,290715.6450,0.826267,147.53020
1,1,0.639451,-16.267598,0.071520,0.444929,0.349414,0.170522,0.651010,164519.5174,0.145400,136.15963
2,2,0.514538,-15.953575,0.110715,0.173699,0.453814,0.029576,0.423865,174495.5667,0.624667,55.31989
3,3,0.734463,-1.357000,0.052965,0.001651,0.159717,0.086366,0.278745,225567.4651,0.487467,147.91212
4,4,0.532968,-13.056437,0.023500,0.068687,0.000001,0.331345,0.477769,213960.6789,0.947333,89.58511


In [3]:
# Checking for missing values
print(train.isna().sum())
print()
print(test.isna().sum())

id                           0
RhythmScore                  0
AudioLoudness                0
VocalContent                 0
AcousticQuality              0
InstrumentalScore            0
LivePerformanceLikelihood    0
MoodScore                    0
TrackDurationMs              0
Energy                       0
BeatsPerMinute               0
dtype: int64

id                           0
RhythmScore                  0
AudioLoudness                0
VocalContent                 0
AcousticQuality              0
InstrumentalScore            0
LivePerformanceLikelihood    0
MoodScore                    0
TrackDurationMs              0
Energy                       0
dtype: int64


In [4]:
train.describe()

,id,RhythmScore,AudioLoudness,VocalContent,AcousticQuality,InstrumentalScore,LivePerformanceLikelihood,MoodScore,TrackDurationMs,Energy,BeatsPerMinute
count,524164.000000,524164.000000,524164.000000,524164.000000,524164.000000,524164.000000,524164.000000,524164.000000,524164.000000,524164.000000,524164.000000
mean,262081.500000,0.632843,-8.379014,0.074443,0.262913,0.117690,0.178398,0.555843,241903.692949,0.500923,119.034899
std,151313.257587,0.156899,4.616221,0.049939,0.223120,0.131845,0.118186,0.225480,59326.601501,0.289952,26.468077
min,0.000000,0.076900,-27.509725,0.023500,0.000005,0.000001,0.024300,0.025600,63973.000000,0.000067,46.718000
25%,131040.750000,0.515850,-11.551933,0.023500,0.069413,0.000001,0.077637,0.403921,207099.876625,0.254933,101.070410
50%,262081.500000,0.634686,-8.252499,0.066425,0.242502,0.074247,0.166327,0.564817,243684.058150,0.511800,118.747660
75%,393122.250000,0.739179,-4.912298,0.107343,0.396957,0.204065,0.268946,0.716633,281851.658500,0.746000,136.686590
max,524163.000000,0.975000,-1.357000,0.256401,0.995000,0.869258,0.599924,0.978000,464723.228100,1.000000,206.037000


In [ ]:
from torch.utils.data import TensorDataset, DataLoader, random_split, Dataset
import numpy as np

class CustomeDataSet(Dataset):
    def __init__(self, x_data, y_data):
        self.x = x_data
        self.y = y_data
        
    def __getitem__(self, index):
        return self.x[index], self.y[index]
    
    def __len__(self):
        return len(self.x)

train_dataset = train.drop(['id'], axis=1)
y_train = train_dataset['BeatsPerMinute']
x_train = train_dataset.drop(['BeatsPerMinute'], axis=1)
y_train = np.array(y_train.values.tolist(), dtype=np.float32)
x_train = np.array(x_train.values.tolist(), dtype=np.float32)

train_dataset = CustomeDataSet(x_train, y_train)
train_dataset, val_dataset = random_split(train_dataset, [0.8, 0.2],)

len(train_dataset), len(val_dataset)

(419332, 104832)

In [ ]:
import torch.nn as nn

class BaselineModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim=1):
        super(BaselineModel, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(in_features=input_dim, out_features=hidden_dim),
            nn.Linear(in_features=hidden_dim, out_features=output_dim)
        )

    def forward(self, x):
        return self.network(x)

In [ ]:
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

In [44]:
input_dim = 9
hidden_dim = 12
output_dim = 1

learning_rate = 5e-12
epochs = 10

In [45]:
train_dataset

In [ ]:
import torch
import torch.optim as optim

model = BaselineModel(input_dim, hidden_dim, output_dim)
optimizer = optim.SGD(model.parameters(), lr=learning_rate)
criterion = nn.MSELoss()

for epoch in range(epochs):
    model.train()
    # Convert numpy array to torch Variable
    inputs = torch.from_numpy(x_train).requires_grad_()
    targets = torch.from_numpy(y_train)
    
    val_inputs = torch.from_numpy(x_val).requires_grad_()
    val_targets = torch.from_numpy(y_val)

    # Clear gradients w.r.t. parameters
    optimizer.zero_grad() 

    # Forward to get output
    outputs = baselineModel(inputs)
    val_outputs = baselineModel(val_inputs)

    # Calculate Loss
    loss = criterion(outputs, targets)
    val_loss = criterion(val_outputs, val_targets)
    
    # Getting gradients w.r.t. parameters
    loss.backward()

    # Updating parameters
    optimizer.step()

    print('epoch {}, loss {}, validation loss {}'.format(epoch, loss.item(), val_loss.item()))

epoch 1, loss 2883512832.0, validation loss 2882125056.0
epoch 2, loss 778913728.0, validation loss 778539136.0
epoch 3, loss 252352192.0, validation loss 252230928.0
epoch 4, loss 85795352.0, validation loss 85754200.0
epoch 5, loss 29624976.0, validation loss 29610812.0
epoch 6, loss 10283686.0, validation loss 10278793.0
epoch 7, loss 3576772.0, validation loss 3575086.25
epoch 8, loss 1245437.625, validation loss 1244861.75
epoch 9, loss 434389.34375, validation loss 434196.40625
epoch 10, loss 152158.125, validation loss 152096.578125
epoch 11, loss 53935.82421875, validation loss 53918.9296875
epoch 12, loss 19751.931640625, validation loss 19749.9453125
epoch 13, loss 7854.83740234375, validation loss 7857.66748046875
epoch 14, loss 3714.12060546875, validation loss 3718.407470703125
epoch 15, loss 2273.185791015625, validation loss 2277.847412109375
epoch 16, loss 1771.6971435546875, validation loss 1776.4122314453125
epoch 17, loss 1597.140380859375, validation loss 1601.82861

In [215]:
class MultipleLayersModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim=1):
        super(MultipleLayersModel, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(in_features=input_dim, out_features=hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(in_features=hidden_dim, out_features=hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(in_features=hidden_dim, out_features=output_dim),
            nn.ReLU()
        )

    def forward(self, x):
        return self.network(x)

In [ ]:
input_dim = 9
hidden_dim = 128
output_dim = 1

learning_rate = 1e-3
epochs = 50

multipleLayersModel = MultipleLayersModel(input_dim, hidden_dim, output_dim)
optimizer_SGD = torch.optim.SGD(multipleLayersModel.parameters(), lr=learning_rate)
optimizer_Adam = torch.optim.Adam(multipleLayersModel.parameters(), lr=learning_rate)

criterion = nn.MSELoss()

In [217]:
for epoch in range(epochs):
    epoch += 1
    # Convert numpy array to torch Variable
    inputs = torch.from_numpy(x_train).requires_grad_()
    targets = torch.from_numpy(y_train)
    
    val_inputs = torch.from_numpy(x_val).requires_grad_()
    val_targets = torch.from_numpy(y_val)

    # Clear gradients w.r.t. parameters
    optimizer_Adam.zero_grad()

    # Forward to get output
    outputs = multipleLayersModel(inputs)
    val_outputs = multipleLayersModel(val_inputs)

    # Calculate Loss
    loss = criterion(outputs, targets)
    val_loss = criterion(val_outputs, val_targets)
    
    # Getting gradients w.r.t. parameters
    loss.backward()

    # Updating parameters
    optimizer_Adam.step()

    print('epoch {}, loss {}, validation loss {}'.format(epoch, loss.item(), val_loss.item()))

epoch 1, loss 25423.63671875, validation loss 24842.85546875
epoch 2, loss 14924.1826171875, validation loss 15015.3173828125
epoch 3, loss 14869.826171875, validation loss 14870.0166015625
epoch 4, loss 14869.826171875, validation loss 14870.0166015625
epoch 5, loss 14869.826171875, validation loss 14870.0166015625
epoch 6, loss 14869.826171875, validation loss 14870.0166015625
epoch 7, loss 14869.826171875, validation loss 14870.0166015625
epoch 8, loss 14869.826171875, validation loss 14870.0166015625
epoch 9, loss 14869.826171875, validation loss 14870.0166015625
epoch 10, loss 14869.826171875, validation loss 14870.0166015625
epoch 11, loss 14869.826171875, validation loss 14870.0166015625
epoch 12, loss 14869.826171875, validation loss 14870.0166015625
epoch 13, loss 14869.826171875, validation loss 14870.0166015625
epoch 14, loss 14869.826171875, validation loss 14870.0166015625
epoch 15, loss 14869.826171875, validation loss 14870.0166015625
epoch 16, loss 14869.826171875, vali

KeyboardInterrupt: 